In [1]:
from pathlib import Path

import pandas as pd

In [2]:
dataset_path = Path("../dataset")
accounts_path = dataset_path / "HI-Small_accounts.csv"
transactions_path = dataset_path / "HI-Small_Trans.csv"
print(f"{accounts_path=}")
print(f"{transactions_path=}")

accounts_path=PosixPath('../dataset/HI-Small_accounts.csv')
transactions_path=PosixPath('../dataset/HI-Small_Trans.csv')


# Accounts

In [3]:
accounts = pd.read_csv(accounts_path)
print(f"{accounts.shape=}")
accounts.head()

accounts.shape=(518581, 5)


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [4]:
accounts['Bank Name'] = accounts['Bank Name'].str.split(' #').str[0]
accounts['Entity Name'] = accounts['Entity Name'].str.split(' #').str[0]

# Transactions

In [5]:
transactions = pd.read_csv(transactions_path)
transactions = transactions.rename(columns={"Account": "From Account", "Account.1": "To Account"})
transactions['Timestamp'] = pd.to_datetime(transactions['Timestamp'])
print(f"{transactions.shape=}")
transactions.head()

transactions.shape=(5078345, 11)


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


# Data Modelling

In [6]:
import math

import networkx as nx
import torch
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.utils import to_networkx

/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vertex (Account)
features:
- bank id
- bank name
- account number
- entity name

Edge (Transaction)
features:
- Amount Received
- Amount Paid
- Receiving Currency
- Payment Currency
- Payment Format

## Vertex

In [7]:
accounts_iterator = accounts.loc[:, ['Bank ID', 'Account Number']].itertuples()

account_to_idx = {(bank, account): i for i, (_, bank, account) in enumerate(accounts_iterator)}

In [8]:
transformer = ColumnTransformer([
    ('categorical_features', OneHotEncoder(sparse_output=False, drop='first'), ['Entity Name'])
], remainder='drop')


vertices = torch.tensor(transformer.fit_transform(accounts), dtype=torch.float)

## Edge

### Connections

In [9]:
transactions.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [10]:
source_keys = list(zip(transactions['From Bank'], transactions['From Account']))
destination_keys = list(zip(transactions['To Bank'], transactions['To Account']))

transactions['source'] = pd.Series(source_keys).map(account_to_idx)
transactions['destination'] = pd.Series(destination_keys).map(account_to_idx)
transactions.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,source,destination
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0,435512,435512
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0,65242,474699
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,65597,65597
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0,475408,475408
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0,475619,475619


In [11]:
transactions[['source', 'destination']].values.T


array([[435512,  65242,  65597, ..., 460201, 430481, 497197],
       [435512, 474699,  65597, ..., 405368, 405368, 405368]],
      shape=(2, 5078345))

In [12]:
edge_index = torch.tensor(transactions[['source', 'destination']].values.T, dtype=torch.long)

### Features

In [13]:
categorical_features = ['Payment Format', 'Receiving Currency', 'Payment Currency']
numerical_features = ['Amount Received', 'Amount Paid']

transformer = ColumnTransformer([
    ('categorical', OneHotEncoder(sparse_output=False), categorical_features),
    ('numerical', RobustScaler(), numerical_features),
], remainder='passthrough', verbose_feature_names_out=False)

transformer.set_output(transform='pandas')

preprocessed_transactions = transformer.fit_transform(transactions)
to_remove = ['From Bank', 'From Account', 'To Account', 'To Bank', 'Is Laundering', 'Timestamp']
preprocessed_transactions = preprocessed_transactions.drop(columns=to_remove)
preprocessed_transactions.columns

Index(['Payment Format_ACH', 'Payment Format_Bitcoin', 'Payment Format_Cash',
       'Payment Format_Cheque', 'Payment Format_Credit Card',
       'Payment Format_Reinvestment', 'Payment Format_Wire',
       'Receiving Currency_Australian Dollar', 'Receiving Currency_Bitcoin',
       'Receiving Currency_Brazil Real', 'Receiving Currency_Canadian Dollar',
       'Receiving Currency_Euro', 'Receiving Currency_Mexican Peso',
       'Receiving Currency_Ruble', 'Receiving Currency_Rupee',
       'Receiving Currency_Saudi Riyal', 'Receiving Currency_Shekel',
       'Receiving Currency_Swiss Franc', 'Receiving Currency_UK Pound',
       'Receiving Currency_US Dollar', 'Receiving Currency_Yen',
       'Receiving Currency_Yuan', 'Payment Currency_Australian Dollar',
       'Payment Currency_Bitcoin', 'Payment Currency_Brazil Real',
       'Payment Currency_Canadian Dollar', 'Payment Currency_Euro',
       'Payment Currency_Mexican Peso', 'Payment Currency_Ruble',
       'Payment Currency_Rupee'

In [14]:
preprocessed_transactions[numerical_features].head()

,Amount Received,Amount Paid
0,0.187976,0.188453
1,-0.116009,-0.116774
2,1.090575,1.094744
3,0.114772,0.114950
4,2.899963,2.911532


In [15]:
preprocessed_transactions.loc[:, ~preprocessed_transactions.columns.isin(numerical_features)]

,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,...,Payment Currency_Rupee,Payment Currency_Saudi Riyal,Payment Currency_Shekel,Payment Currency_Swiss Franc,Payment Currency_UK Pound,Payment Currency_US Dollar,Payment Currency_Yen,Payment Currency_Yuan,source,destination
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,435512,435512
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,65242,474699
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,65597,65597
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,475408,475408
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,475619,475619
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,228536,405368
5078341,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,162963,405368
5078342,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460201,405368
5078343,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,430481,405368


In [16]:
preprocessed_transactions.columns.difference(numerical_features, sort=False)

Index(['Payment Format_ACH', 'Payment Format_Bitcoin', 'Payment Format_Cash',
       'Payment Format_Cheque', 'Payment Format_Credit Card',
       'Payment Format_Reinvestment', 'Payment Format_Wire',
       'Receiving Currency_Australian Dollar', 'Receiving Currency_Bitcoin',
       'Receiving Currency_Brazil Real', 'Receiving Currency_Canadian Dollar',
       'Receiving Currency_Euro', 'Receiving Currency_Mexican Peso',
       'Receiving Currency_Ruble', 'Receiving Currency_Rupee',
       'Receiving Currency_Saudi Riyal', 'Receiving Currency_Shekel',
       'Receiving Currency_Swiss Franc', 'Receiving Currency_UK Pound',
       'Receiving Currency_US Dollar', 'Receiving Currency_Yen',
       'Receiving Currency_Yuan', 'Payment Currency_Australian Dollar',
       'Payment Currency_Bitcoin', 'Payment Currency_Brazil Real',
       'Payment Currency_Canadian Dollar', 'Payment Currency_Euro',
       'Payment Currency_Mexican Peso', 'Payment Currency_Ruble',
       'Payment Currency_Rupee'

In [17]:
edge_features = torch.tensor(preprocessed_transactions.values, dtype=torch.float)
print(edge_features.shape)
print(edge_features[:2])


torch.Size([5078345, 41])
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  1.8798e-01,  1.8845e-01,  4.3551e+05,
          4.3551e+05],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0

### Label

In [18]:
edge_label = transactions['Is Laundering'].values
edge_label

array([0, 0, 0, ..., 0, 0, 0], shape=(5078345,))

## Graph

### Splitting Data

In [ ]:
train_cut_off_index = math.floor(transactions.shape[0] * 0.7)
train_cut_off_time = transactions['Timestamp'].sort_values().iloc[train_cut_off_index]
train_edges_index = transactions[transactions['Timestamp'] <= train_cut_off_time].index
train_edges_mask = transactions.index.isin(train_edges_index)
print(f'train proportion: {train_edges_mask.sum() / transactions.shape[0]}')

train proportion: 0.7000227436300606


In [ ]:
validation_cut_off_index = train_cut_off_index + math.floor(transactions.shape[0] * 0.15)
validation_cut_off_time = transactions['Timestamp'].sort_values().iloc[validation_cut_off_index]
validation_edges_index = transactions[(transactions['Timestamp'] <= validation_cut_off_time) & (transactions['Timestamp'] > train_cut_off_time)].index
validation_edges_mask = transactions.index.isin(validation_edges_index)
print(f'validation proportion: {validation_edges_mask.sum() / transactions.shape[0]}')

validation proportion: 0.14999945848499857


In [ ]:
test_edges_index = transactions[transactions['Timestamp'] > validation_cut_off_time].index
test_edges_mask = transactions.index.isin(test_edges_index)
print(f'test proportion: {test_edges_mask.sum() / transactions.shape[0]}')

test proportion: 0.14997779788494087


In [ ]:
device = torch.device('cpu')
unix_timestamp = transactions['Timestamp'].astype('int64') // 10**9
data = Data(x=vertices, edge_index=edge_index, edge_attr=edge_features)
data.edge_time = torch.tensor(unix_timestamp.values)
data.train_mask = torch.tensor(train_edges_mask)

In [ ]:
train_edge_label = torch.tensor(transactions.iloc[train_edges_mask]['Is Laundering'].values, dtype=torch.float)
train_transactions_time = torch.tensor(unix_timestamp.iloc[train_edges_mask].values, dtype=torch.long)
indices = torch.tensor(train_edges_index, dtype=torch.long)
edge_label_index = data.edge_index[:, indices]
train_loader = LinkNeighborLoader(
    data,
    num_neighbors=[10, 5],
    batch_size=128,
    edge_label_index=edge_label_index,
    edge_label=train_edge_label,
    edge_label_time=train_transactions_time,
    time_attr='edge_time',
)

In [ ]:
batch = next(iter(train_loader))
batch

Data(x=[539, 5], edge_index=[2, 571], edge_attr=[571, 41], edge_time=[571], train_mask=[571], n_id=[539], e_id=[571], batch=[539], num_sampled_nodes=[3], num_sampled_edges=[2], input_id=[128], edge_label_index=[2, 128], edge_label=[128], edge_label_time=[128])

In [ ]:
validation_edge_label = torch.tensor(transactions.iloc[validation_edges_mask]['Is Laundering'].values, dtype=torch.float)
validation_transactions_time = torch.tensor(unix_timestamp.iloc[validation_edges_mask].values, dtype=torch.long)
indices = torch.tensor(validation_edges_index, dtype=torch.long)
edge_label_index = data.edge_index[:, indices]
validation_loader = LinkNeighborLoader(
    data,
    num_neighbors=[10, 5],
    batch_size=128,
    edge_label_index=edge_label_index,
    edge_label=validation_edge_label,
    edge_label_time=validation_transactions_time,
    time_attr='edge_time',
    num_workers=0,
    # Keep loading on the main process to avoid duplicating graph memory on macOS.
    persistent_workers=False
)

# Model

In [ ]:
from datetime import datetime
import random
from time import perf_counter
from typing import Literal

import numpy as np
import torch
from sklearn.metrics import classification_report
from tqdm import tqdm
from torch_geometric.nn import GAT, TGNMemory, TransformerConv, GCNConv
from torch_geometric.nn.models.tgn import (
    LastAggregator,
    IdentityMessage,
    LastNeighborLoader,
)
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

In [ ]:
class EdgeClassifierGAT(torch.nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_channels, num_layers, output_dim):
        super().__init__()
        self.backbone = GAT(
            in_channels=in_channels,
            edge_dim=edge_dim,
            hidden_channels=hidden_channels,
            num_layers=num_layers,
            out_channels=hidden_channels,
            v2=True,
        )
        self.classifier = torch.nn.Linear(2 * hidden_channels, output_dim)

    def forward(self, x, edge_index, edge_label_index, edge_attr):
        h = self.backbone(x, edge_index, edge_attr=edge_attr)

        row, col = edge_label_index

        edge_features = torch.cat([h[row], h[col]], dim=-1)
        edge_features = edge_features.relu()

        return self.classifier(edge_features)

In [31]:
def set_seed(seed) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    torch.use_deterministic_algorithms(True)

In [ ]:
class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_loader,
        val_loader,
        criterion,
        Optimizer: type[torch.optim.Optimizer],
        device: Literal['cpu', 'mps'] = "cpu",
    ) -> None:
        self.device = device

        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = Optimizer(
            model.parameters(),
            lr=0.0001,
        )
        self.criterion = criterion.to(self.device)

        self.epochs_run = 0

    @staticmethod
    def _logits_to_pred(logits: torch.Tensor) -> torch.Tensor:
        # BCEWithLogitsLoss uses raw logits; threshold at 0 == sigmoid 0.5.
        return (logits.squeeze(-1) > 0).to(torch.int64)

    def _run_batch(self, batch):
        batch = batch.to(self.device)
        self.optimizer.zero_grad()

        pred = self.model(batch.x, batch.edge_index, batch.edge_label_index, edge_attr=batch.edge_attr)

        loss = self.criterion(pred, batch.edge_label.unsqueeze(-1).to(self.device))

        # Keep only batch tensors for metrics to avoid Python list growth.
        batch_pred = self._logits_to_pred(pred).detach().cpu()
        batch_true = batch.edge_label.to(torch.int64).detach().cpu()

        loss.backward()
        self.optimizer.step()

        return loss.item(), batch_pred, batch_true

    def _run_epoch(self, epoch: int):
        progress_bar = tqdm(enumerate(self.train_loader), total=len(self.train_loader))
        loss = 0.0
        epoch_pred = []
        epoch_true = []

        for i, batch in progress_bar:
            loss_val, batch_pred, batch_true = self._run_batch(batch)
            loss += loss_val
            epoch_pred.append(batch_pred)
            epoch_true.append(batch_true)
            progress_bar.set_postfix({"loss": f"{(loss / (i + 1)):.4f}"})

        avg_loss = loss / len(self.train_loader)
        train_pred = torch.cat(epoch_pred).numpy()
        train_true = torch.cat(epoch_true).numpy()

        return avg_loss, train_true, train_pred

    @torch.no_grad()
    def _evaluate(self, epoch):
        loss = 0.0
        epoch_pred = []
        epoch_true = []

        for i, batch in enumerate(self.val_loader):
            batch = batch.to(self.device)
            pred = self.model(batch.x, batch.edge_index, batch.edge_label_index, batch.edge_attr)
            batch_loss = self.criterion(pred, batch.edge_label.unsqueeze(-1))

            epoch_pred.append(self._logits_to_pred(pred).detach().cpu())
            epoch_true.append(batch.edge_label.to(torch.int64).detach().cpu())
            loss += batch_loss.item()

        avg_loss = loss / len(self.val_loader)
        val_pred = torch.cat(epoch_pred).numpy()
        val_true = torch.cat(epoch_true).numpy()

        return avg_loss, val_true, val_pred

    @staticmethod
    def _log_mps_memory(tag: str) -> None:
        if not torch.backends.mps.is_available():
            return
        torch.mps.synchronize()
        current_gb = torch.mps.current_allocated_memory() / (1024 ** 3)
        driver_gb = torch.mps.driver_allocated_memory() / (1024 ** 3)
        print(f"[{tag}] MPS current={current_gb:.2f} GB | driver={driver_gb:.2f} GB")

    def train(self, max_epochs: int) -> None:
        run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
        with SummaryWriter(f"logs/tensorboard/{run_id}") as writer:
            for epoch in range(self.epochs_run, max_epochs):
                start_time = perf_counter()
                self.model.train()
                avg_loss, train_true, train_pred = self._run_epoch(epoch)
                time_elapsed = perf_counter() - start_time

                writer.add_scalar("Loss/train", avg_loss, epoch)
                print(f"Epoch: {epoch} | Time: {time_elapsed:.2f}s | Loss: {avg_loss:.3f}")
                print(classification_report(train_true, train_pred, digits=5))

                self.model.eval()
                val_loss, val_true, val_pred = self._evaluate(epoch)
                print(f"Epoch: {epoch} | Validation Loss: {val_loss:.3f}")
                print(classification_report(val_true, val_pred, digits=5))
                writer.add_scalar("Loss/validation", val_loss, epoch)

                # self._log_mps_memory(f"epoch={epoch}")
                if torch.backends.mps.is_available():
                    torch.mps.empty_cache()

                self.epochs_run += 1

In [ ]:
set_seed(124)
model = EdgeClassifierGAT(in_channels=5, edge_dim=edge_features.shape[-1], hidden_channels=32, num_layers=2, output_dim=1)
optimizer_cls = torch.optim.Adam
# label_weights = 1e5 / torch.tensor(transactions['Is Laundering'].value_counts().values, dtype=torch.float)
value_counts = transactions['Is Laundering'].value_counts().values
weight = torch.tensor([value_counts[0] / value_counts[1]], dtype=torch.float)
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=validation_loader,
    Optimizer=optimizer_cls,
    criterion=torch.nn.BCEWithLogitsLoss(pos_weight=weight),
    device=device
)
trainer.train(max_epochs=5)

100%|██████████| 27774/27774 [01:37<00:00, 283.69it/s, loss=1.2421]


Epoch: 0 | Time: 97.96s | Loss: 1.242
              precision    recall  f1-score   support

           0    0.99924   0.89550   0.94453   3552101
           1    0.00121   0.15686   0.00239      2856

    accuracy                        0.89490   3554957
   macro avg    0.50022   0.52618   0.47346   3554957
weighted avg    0.99844   0.89490   0.94377   3554957



/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

Epoch: 0 | Validation Loss: 1.464
              precision    recall  f1-score   support

           0    0.99900   1.00000   0.99950    760989
           1    0.00000   0.00000   0.00000       760

    accuracy                        0.99900    761749
   macro avg    0.49950   0.50000   0.49975    761749
weighted avg    0.99801   0.99900   0.99850    761749



 38%|███▊      | 10612/27774 [00:25<00:41, 414.16it/s, loss=1.0627]


KeyboardInterrupt: 

# Temporal

In [ ]:
import random
from typing import Literal
from datetime import datetime

from torch.nn import Linear
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch_geometric.data import TemporalData
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.nn import TransformerConv, TGNMemory
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastAggregator,
    LastNeighborLoader,
)

In [21]:
device = torch.device('mps')
unix_timestamp = transactions['Timestamp'].astype('int64') // 10**9
data = TemporalData(
    src=edge_index[0],
    dst=edge_index[1],
    t=torch.tensor(unix_timestamp.values),
    msg=edge_features,
    y=torch.tensor(transactions['Is Laundering'].values, dtype=torch.float)
).to(device)

In [22]:
train_data, val_data, test_data = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)

# Keep natural class labels from the dataset; do not inject synthetic negatives.
train_loader = TemporalDataLoader(train_data, batch_size=512)
val_loader = TemporalDataLoader(val_data, batch_size=512)
test_loader = TemporalDataLoader(test_data, batch_size=512)

neighbor_loader = LastNeighborLoader(data.num_nodes, size=15, device=device)

In [23]:
class GraphAttentionEmbedding(torch.nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        edge_dim = msg_dim + time_enc.out_channels
        self.conv = TransformerConv(in_channels, out_channels // 2, heads=2,
                                    dropout=0.1, edge_dim=edge_dim)

    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = last_update[edge_index[0]] - t
        rel_t_enc = self.time_enc(rel_t.to(x.dtype))
        edge_attr = torch.cat([rel_t_enc, msg], dim=-1)
        return self.conv(x, edge_index, edge_attr)
    

# Transformed LinkPredictor into EdgeClassifier
class EdgeClassifier(torch.nn.Module):
    def __init__(self, in_channels, edge_dim):
        super().__init__()
        # Concatenate src_node, dst_node, and raw edge_attr
        self.lin = Linear(in_channels * 2 + edge_dim, in_channels)
        self.lin_final = Linear(in_channels, 1)

    def forward(self, z_src, z_dst, edge_attr):
        h = torch.cat([z_src, z_dst, edge_attr], dim=-1)
        h = self.lin(h).relu()
        return self.lin_final(h)

In [ ]:
class TGNTrainer:
    def __init__(
        self,
        memory: TGNMemory,
        gnn: torch.nn.Module,
        edge_classifier: torch.nn.Module,
        data: TemporalData,
        neighbor_loader,
        train_loader,
        val_loader,
        criterion,
        Optimizer: type[torch.optim.Optimizer],
        device: Literal['cpu', 'mps'] = "cpu",
    ) -> None:
        self.device = device

        self.memory = memory.to(self.device)
        self.gnn = gnn.to(self.device)
        self.edge_classifier = edge_classifier.to(self.device)

        self.data = data
        self.neighbor_loader = neighbor_loader
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = Optimizer(
            set(self.memory.parameters()) | set(self.gnn.parameters()) | set(self.edge_classifier.parameters()),
            lr=0.0001,
        )
        self.criterion = criterion.to(self.device)

        self.epochs_run = 0
        self.assoc = torch.empty(data.num_nodes, dtype=torch.long, device=self.device)

    @staticmethod
    def _logits_to_pred(logits: torch.Tensor) -> torch.Tensor:
        # BCEWithLogitsLoss uses raw logits; threshold at 0 == sigmoid 0.5.
        return (logits.squeeze(-1) > 0).to(torch.int64)

    def _run_batch(self, batch):
        batch = batch.to(self.device)
        self.optimizer.zero_grad()

        n_id, edge_index, e_id = self.neighbor_loader(batch.n_id)
        self.assoc[n_id] = torch.arange(n_id.size(0), device=self.device)

        z, last_update = self.memory(n_id)
        z = self.gnn(
            z,
            last_update,
            edge_index,
            self.data.t[e_id].to(self.device),
            self.data.msg[e_id].to(self.device)
        )

        out = self.edge_classifier(z[self.assoc[batch.src]], z[self.assoc[batch.dst]], batch.msg)
        
        loss = self.criterion(out.squeeze(), batch.y.float())

        self.memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        self.neighbor_loader.insert(batch.src, batch.dst)

        loss.backward()
        self.optimizer.step()
        self.memory.detach()

        return loss.item() * batch.num_events, batch.num_events, self._logits_to_pred(out).detach().cpu(), batch.y.to(torch.int64).detach().cpu()

    def _run_epoch(self, epoch: int):
        progress_bar = tqdm(enumerate(self.train_loader), total=len(self.train_loader))
        loss = 0.0
        num_events = 0
        epoch_pred = []
        epoch_true = []

        for i, batch in progress_bar:
            loss_val, batch_num_events, batch_pred, batch_true = self._run_batch(batch)
            loss += loss_val
            num_events += batch_num_events

            epoch_pred.append(batch_pred)
            epoch_true.append(batch_true)
            progress_bar.set_postfix({"loss": f"{(loss/num_events):.4f}"})


        avg_loss = loss / num_events
        train_pred = torch.cat(epoch_pred).numpy()
        train_true = torch.cat(epoch_true).numpy()
        # print(f"{train_pred=}")
        # print(f"{train_true=}")

        return avg_loss, train_true, train_pred

    @torch.no_grad()
    def _evaluate(self, epoch):
        loss = 0.0
        epoch_pred = []
        epoch_true = []

        for i, batch in enumerate(self.val_loader):
            batch = batch.to(self.device)
            pred = self.model(batch.x, batch.edge_index, batch.edge_label_index, batch.edge_attr)
            batch_loss = self.criterion(pred, batch.edge_label.unsqueeze(-1))

            epoch_pred.append(self._logits_to_pred(pred).detach().cpu())
            epoch_true.append(batch.edge_label.to(torch.int64).detach().cpu())
            loss += batch_loss.item()

        avg_loss = loss / len(self.val_loader)
        val_pred = torch.cat(epoch_pred).numpy()
        val_true = torch.cat(epoch_true).numpy()

        return avg_loss, val_true, val_pred

    @staticmethod
    def _log_mps_memory(tag: str) -> None:
        if not torch.backends.mps.is_available():
            return
        torch.mps.synchronize()
        current_gb = torch.mps.current_allocated_memory() / (1024 ** 3)
        driver_gb = torch.mps.driver_allocated_memory() / (1024 ** 3)
        print(f"[{tag}] MPS current={current_gb:.2f} GB | driver={driver_gb:.2f} GB")

    def eval_mode(self):
        self.memory.eval()
        self.gnn.eval()
        self.edge_classifier.eval()
    
    def train_mode(self):
        self.memory.train()
        self.gnn.train()
        self.edge_classifier.train()

    def _reset_temporal_state(self) -> None:
        # Ensure edge ids from neighbor sampling start from the current split each epoch.
        self.memory.reset_state()
        self.neighbor_loader.reset_state()

    def train(self, max_epochs: int) -> None:
        run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
        with SummaryWriter(f"logs/tensorboard/{run_id}") as writer:
            for epoch in range(self.epochs_run, max_epochs):
                self._reset_temporal_state()
                start_time = perf_counter()
                self.train_mode()
                avg_loss, train_true, train_pred = self._run_epoch(epoch)
                time_elapsed = perf_counter() - start_time

                writer.add_scalar("Loss/train", avg_loss, epoch)
                print(f"Epoch: {epoch} | Time: {time_elapsed:.2f}s | Loss: {avg_loss:.3f}")
                print(classification_report(train_true, train_pred, digits=5))

                # self.eval_mode()
                # val_loss, val_true, val_pred = self._evaluate(epoch)
                # print(f"Epoch: {epoch} | Validation Loss: {val_loss:.3f}")
                # # print(classification_report(val_true, val_pred, digits=5))
                # writer.add_scalar("Loss/validation", val_loss, epoch)

                # self._log_mps_memory(f"epoch={epoch}")
                # if torch.backends.mps.is_available():
                #     torch.mps.empty_cache()

                self.epochs_run += 1

In [34]:
set_seed(124)

embedding_dim = memory_dim = time_dim = 64
memory = TGNMemory(
    data.num_nodes,
    data.msg.size(-1),
    memory_dim,
    time_dim,
    message_module=IdentityMessage(data.msg.size(-1), memory_dim, time_dim),
    aggregator_module=LastAggregator(),
)

gnn = GraphAttentionEmbedding(
    in_channels=memory_dim,
    out_channels=embedding_dim,
    msg_dim=data.msg.size(-1),
    time_enc=memory.time_enc,
)

edge_classifier = EdgeClassifier(in_channels=embedding_dim, edge_dim=data.msg.size(-1))

Optimizer = torch.optim.Adam
value_counts = transactions['Is Laundering'].value_counts().values
weight = torch.tensor([500], dtype=torch.float)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=weight)

trainer = TGNTrainer(
    memory=memory,
    gnn=gnn,
    edge_classifier=edge_classifier,
    data=data,
    neighbor_loader=neighbor_loader,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    Optimizer=Optimizer,
    device=device
)
trainer.train(max_epochs=5)

NameError: name 'np' is not defined